# Offline RAG processing

Read the local knowledge base, split it into chunks, generate embeddings locally with Qwen, and persist them in Chroma. No OpenAI or SiliconFlow API key is required.

In [ ]:
from pathlib import Path
import hashlib

import torch
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

In [ ]:
# Support running the notebook from the repository root, rag/, or this folder.
candidate_directories = [
    Path.cwd(),
    Path.cwd() / 'rag/other_rag_models/native_rag',
    Path.cwd() / 'other_rag_models/native_rag',
]

base_directory = next(
    (path.resolve() for path in candidate_directories if (path / 'knowledge_base.txt').exists()),
    None,
)

if base_directory is None:
    raise FileNotFoundError('Could not locate knowledge_base.txt')

knowledge_base_path = base_directory / 'knowledge_base.txt'
persist_directory = base_directory / 'chroma_db_qwen'

print('Knowledge base:', knowledge_base_path)
print('Chroma directory:', persist_directory)

In [ ]:
text = knowledge_base_path.read_text(encoding='utf-8')
documents = [
    Document(
        page_content=text,
        metadata={'source': str(knowledge_base_path)},
    )
]

print(f'Loaded {len(documents)} document with {len(text)} characters.')

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
)

splits = text_splitter.split_documents(documents)
print(f'Created {len(splits)} chunks.')

if splits:
    print(splits[0].page_content[:300])

In [ ]:
if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'

embeddings = HuggingFaceEmbeddings(
    model_name='Qwen/Qwen3-Embedding-0.6B',
    model_kwargs={'device': device},
    encode_kwargs={'normalize_embeddings': True},
)

print('Embedding device:', device)

In [ ]:
# Use stable IDs so rerunning this notebook updates the same chunks instead of duplicating them.
document_ids = [
    hashlib.sha256(f'{index}:{document.page_content}'.encode('utf-8')).hexdigest()
    for index, document in enumerate(splits)
]

vectorstore = Chroma(
    collection_name='native_rag_qwen',
    embedding_function=embeddings,
    persist_directory=str(persist_directory),
)

vectorstore.add_documents(documents=splits, ids=document_ids)
print(f'Stored {len(splits)} chunks in {persist_directory}.')

In [ ]:
query = 'What information is in the knowledge base?'
results = vectorstore.similarity_search(query, k=min(3, len(splits)))

for index, result in enumerate(results, start=1):
    print(f'--- Result {index} ---')
    print(result.page_content)
    print(result.metadata)